# CIC-IDS2017 Thursday: NFStream extraction and daily labeling

This notebook extracts bidirectional network flows from the original CIC-IDS2017 Thursday PCAP capture using NFStream. It cleans the extracted records, creates the timestamp used by the original labeling procedure, applies the day-specific attack rules, assigns the remaining flows to the benign class, removes duplicate rows, and exports the daily CSV consumed by the GenIDS-CIC17 consolidation notebook.

Only the input and output paths in the configuration cell should be changed. The original PCAP capture is not distributed with this repository.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_FILE = Path("/path/to/cic-ids2017/pcaps/04_thursday.pcap")
OUTPUT_DIR = Path("/path/to/output/nfstream_daily_csv")
OUTPUT_FILE = OUTPUT_DIR / "04_nfstream_thursday.csv"

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 5
STATISTICAL_ANALYSIS = True
DECODE_TUNNELS = True
BPF_FILTER = 'ip'
TIMEZONE = "America/Moncton"

DROP_COLUMNS = [
    "content_type",
    "user_agent",
    "server_fingerprint",
    "client_fingerprint",
    "requested_server_name",
]

## 2. Imports and helper functions

In [ ]:
import pandas as pd
import pytz
import nfstream
from nfstream import NFStreamer


def validate_input(pcap_file):
    if not pcap_file.is_file():
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")


def extract_flows(pcap_file):
    return NFStreamer(
        source=str(pcap_file),
        idle_timeout=IDLE_TIMEOUT,
        active_timeout=ACTIVE_TIMEOUT,
        statistical_analysis=STATISTICAL_ANALYSIS,
        decode_tunnels=DECODE_TUNNELS,
        bpf_filter=BPF_FILTER,
    ).to_pandas()


def prepare_flows(frame):
    prepared = frame.drop(columns=DROP_COLUMNS, errors="ignore").dropna().copy()
    if "src2dst_first_seen_ms" not in prepared.columns:
        raise ValueError("NFStream output is missing the src2dst_first_seen_ms column.")
    timestamps = pd.to_datetime(
        prepared["src2dst_first_seen_ms"], unit="ms", utc=True
    ).dt.tz_convert(pytz.timezone(TIMEZONE))
    prepared["Timestamp"] = timestamps.dt.strftime("%d/%m/%Y %I:%M")
    prepared["binary"] = ""
    prepared["multiclass"] = ""
    return prepared.reset_index(drop=True)


def class_summary(frame, column):
    return pd.DataFrame({
        "count": frame[column].value_counts(),
        "percentage": frame[column].value_counts(normalize=True).mul(100).round(2),
    })


def save_daily_flows(frame, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_file, index=False)
    if not output_file.is_file():
        raise OSError(f"The output file was not created: {output_file}")

## 3. Validate the input and extract NFStream flows

In [ ]:
print(f"NFStream version: {nfstream.__version__}")
validate_input(PCAP_FILE)

raw_flows = extract_flows(PCAP_FILE)
print(f"Extracted flows: {len(raw_flows):,}")
print(f"Extracted columns: {raw_flows.shape[1]}")

## 4. Clean and prepare the extracted flows

In [ ]:
daily_flows = prepare_flows(raw_flows)

print(f"Flows after cleaning: {len(daily_flows):,}")
print(f"Rows removed during cleaning: {len(raw_flows) - len(daily_flows):,}")

## 5. Apply the Thursday labeling rules

In [ ]:
df = daily_flows

mask_webattack_bruteforce = ((df["src_ip"] == '172.16.0.1') &
                             (df["dst_ip"] == '192.168.10.50') &
                             ((df['Timestamp'] == '06/07/2017 09:24') |
                              (df['Timestamp'] == '06/07/2017 09:23') |
                              (df['Timestamp'] == '06/07/2017 09:53') |
                              (df['Timestamp'] == '06/07/2017 09:58') |
                              (df['Timestamp'] == '06/07/2017 09:59') |
                              (df['Timestamp'] == '06/07/2017 09:01') |
                              (df['Timestamp'] == '06/07/2017 09:47') |
                              (df['Timestamp'] == '06/07/2017 09:51') |
                              (df['Timestamp'] == '06/07/2017 09:48') |
                              (df['Timestamp'] == '06/07/2017 09:36') |
                              (df['Timestamp'] == '06/07/2017 09:41') |
                              (df['Timestamp'] == '06/07/2017 09:56') |
                              (df['Timestamp'] == '06/07/2017 09:34') |
                              (df['Timestamp'] == '06/07/2017 09:32') |
                              (df['Timestamp'] == '06/07/2017 09:29') |
                              (df['Timestamp'] == '06/07/2017 09:26') |
                              (df['Timestamp'] == '06/07/2017 09:33') |
                              (df['Timestamp'] == '06/07/2017 09:39') |
                              (df['Timestamp'] == '06/07/2017 09:42') |
                              (df['Timestamp'] == '06/07/2017 09:35') |
                              (df['Timestamp'] == '06/07/2017 09:22') |
                              (df['Timestamp'] == '06/07/2017 09:55') |
                              (df['Timestamp'] == '06/07/2017 09:21') |
                              (df['Timestamp'] == '06/07/2017 10:00') |
                              (df['Timestamp'] == '06/07/2017 09:19') |
                              (df['Timestamp'] == '06/07/2017 09:18') |
                              (df['Timestamp'] == '06/07/2017 09:16') |
                              (df['Timestamp'] == '06/07/2017 09:56') |
                              (df['Timestamp'] == '06/07/2017 09:40') |
                              (df['Timestamp'] == '06/07/2017 09:57') |
                              (df['Timestamp'] == '06/07/2017 09:54') |
                              (df['Timestamp'] == '06/07/2017 09:46') |
                              (df['Timestamp'] == '06/07/2017 09:43') |
                              (df['Timestamp'] == '06/07/2017 09:25') |
                              (df['Timestamp'] == '06/07/2017 09:52') |
                              (df['Timestamp'] == '06/07/2017 09:31') |
                              (df['Timestamp'] == '06/07/2017 09:38') |
                              (df['Timestamp'] == '06/07/2017 09:37') |
                              (df['Timestamp'] == '06/07/2017 09:44') |
                              (df['Timestamp'] == '06/07/2017 09:45') |
                              (df['Timestamp'] == '06/07/2017 09:50') |
                              (df['Timestamp'] == '06/07/2017 09:49') |
                              (df['Timestamp'] == '06/07/2017 09:30') |
                              (df['Timestamp'] == '06/07/2017 09:28') |
                              (df['Timestamp'] == '06/07/2017 09:27') |
                              (df['Timestamp'] == '06/07/2017 09:15')))


df.loc[mask_webattack_bruteforce, df.multiclass.name] = 'webattack_bruteforce'
df.loc[mask_webattack_bruteforce, df.binary.name] = 'malign'

mask_webattack_xss = ((df["src_ip"] == '172.16.0.1') &
                      (df["dst_ip"] == '192.168.10.50') &
                      ((df['Timestamp'] == '06/07/2017 10:24') |
                       (df['Timestamp'] == '06/07/2017 10:22') |
                       (df['Timestamp'] == '06/07/2017 10:21') |
                       (df['Timestamp'] == '06/07/2017 10:30') |
                       (df['Timestamp'] == '06/07/2017 10:27') |
                       (df['Timestamp'] == '06/07/2017 10:34') |
                       (df['Timestamp'] == '06/07/2017 10:33') |
                       (df['Timestamp'] == '06/07/2017 10:20') |
                       (df['Timestamp'] == '06/07/2017 10:19') |
                       (df['Timestamp'] == '06/07/2017 10:26') |
                       (df['Timestamp'] == '06/07/2017 10:18') |
                       (df['Timestamp'] == '06/07/2017 10:31') |
                       (df['Timestamp'] == '06/07/2017 10:25') |
                       (df['Timestamp'] == '06/07/2017 10:23') |
                       (df['Timestamp'] == '06/07/2017 10:29') |
                       (df['Timestamp'] == '06/07/2017 10:17') |
                       (df['Timestamp'] == '06/07/2017 10:28') |
                       (df['Timestamp'] == '06/07/2017 10:32') |
                       (df['Timestamp'] == '06/07/2017 10:16') |
                       (df['Timestamp'] == '06/07/2017 10:35') |
                       (df['Timestamp'] == '06/07/2017 10:15')))


df.loc[mask_webattack_xss, df.multiclass.name] = 'webattack_xss'
df.loc[mask_webattack_xss, df.binary.name] = 'malign'

mask_webattack_sql_injection = ((df["src_ip"] == '172.16.0.1') &
                                (df["dst_ip"] == '192.168.10.50') &
                                ((df['Timestamp'] == '06/07/2017 10:40') |
                                 (df['Timestamp'] == '06/07/2017 10:41') |
                                 (df['Timestamp'] == '06/07/2017 10:42')))


df.loc[mask_webattack_sql_injection, df.multiclass.name] = 'webattack_sql_injection'
df.loc[mask_webattack_sql_injection, df.binary.name] = 'malign'

mask_infiltration = ((df["src_ip"] == '192.168.10.8') &
                     (df["dst_ip"] == '205.174.165.73'))


df.loc[mask_infiltration, df.multiclass.name] = 'infiltration'
df.loc[mask_infiltration, df.binary.name] = 'malign'

df.loc[(df['binary'] == ''), df.binary.name] = 'benign'
df.loc[(df['multiclass'] == ''), df.multiclass.name] = 'benign'

daily_flows = df

## 6. Remove duplicates and summarize the daily dataset

In [ ]:
rows_before = len(daily_flows)
daily_flows = daily_flows.drop_duplicates().reset_index(drop=True)

unlabeled = daily_flows[["binary", "multiclass"]].eq("").any(axis=1).sum()
if unlabeled:
    raise ValueError(f"Found {unlabeled} flows without complete labels.")

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal:  {len(daily_flows):,}")
display(class_summary(daily_flows, "binary"))
display(class_summary(daily_flows, "multiclass"))

## 7. Export the daily flow file

In [ ]:
save_daily_flows(daily_flows, OUTPUT_FILE)

print(f"Daily dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {daily_flows.shape}")
display(daily_flows.head())